In [3]:
!pip install dlib
!pip install imutils
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2
!wget https://www.soundjay.com/buttons/sounds/button-3.wav -O alarm.wav
# Import necessary libraries
from scipy.spatial import distance
from imutils import face_utils
import imutils
import dlib
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, Audio
from google.colab.output import eval_js
from base64 import b64decode

# --- JavaScript and Python Functions for Webcam Access ---

def video_stream():
  """
  Starts the webcam stream and displays it in the Colab output.
  """
  js = Javascript('''
    var video;
    var div = null;
    var stream;

    // Starts the video stream and creates the necessary DOM elements
    async function startVideoStream() {
      // Create a container div
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      // Add a label
      const instruction = document.createElement('div');
      instruction.innerHTML = "Click on the video stream to stop.";
      div.appendChild(instruction);

      // Create the video element
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { stopVideoStream(); }; // Stop stream on click
      div.appendChild(video);

      // Get webcam access
      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Store variables on the window object to be globally accessible
      window.video = video;
      window.stream = stream;
      window.div = div;
    }

    // Stops the video stream and removes the DOM elements
    function stopVideoStream() {
      if (window.stream) {
        window.stream.getTracks().forEach(track => track.stop());
      }
      if (window.div) {
        window.div.remove();
      }
      window.video = null;
      window.stream = null;
      window.div = null;
    }

    // Start the stream automatically
    startVideoStream();
  ''')
  display(js)

def video_frame():
  """
  Captures a single frame from the webcam stream.
  """
  js_code = '''
    async function captureFrame() {
      // Return null if the video stream is not active
      if (!window.video) {
        return null;
      }
      // Create a canvas to draw the frame on
      const canvas = document.createElement('canvas');
      canvas.width = window.video.videoWidth;
      canvas.height = window.video.videoHeight;
      const context = canvas.getContext('2d');
      context.drawImage(window.video, 0, 0, canvas.width, canvas.height);

      // Return the frame as a data URL
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    captureFrame();
  '''
  try:
    data_url = eval_js(js_code)
    if data_url is None:
      return None

    # Decode the base64 image data
    header, encoded = data_url.split(',', 1)
    img_bytes = b64decode(encoded)
    np_arr = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return frame
  except Exception as e:
    # This can happen if the user stops the stream
    return None

def video_release():
    """Releases the webcam and cleans up the output."""
    eval_js('stopVideoStream()')

def eye_aspect_ratio(eye):
    """Computes the eye aspect ratio (EAR)."""
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear
    # --- Constants and Initializations ---
# ADJUST THESE VALUES FOR YOUR FACE AND LIGHTING
thresh = 0.25
# Lowered for easier testing. It now only needs 5 frames of closed eyes.
frame_check = 5
# Counter for consecutive frames with closed eyes
flag = 0

detect = dlib.get_frontal_face_detector()
predict = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["left_eye"]
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["right_eye"]

import time

try:
    # Start the webcam stream
    video_stream()
    print("Video stream starting... Please wait for initialization.")

    # --- Wait for the first frame to ensure the webcam is ready ---
    frame = None
    for i in range(10): # Try for 5 seconds
        frame = video_frame()
        if frame is not None:
            break
        time.sleep(0.5)

    if frame is None:
        raise RuntimeError("Could not start webcam. Please check browser permissions and rerun the cells.")

    print("Initialization complete. Drowsiness detection is running...")
    print("Click on the video stream to stop.")

    # --- Main Loop ---
    while True:
        # Pre-processing
        frame = imutils.resize(frame, width=450)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        subjects = detect(gray, 0)

        for subject in subjects:
            shape = predict(gray, subject)
            shape = face_utils.shape_to_np(shape)

            leftEye = shape[lStart:lEnd]
            rightEye = shape[rStart:rEnd]

            leftEAR = eye_aspect_ratio(leftEye)
            rightEAR = eye_aspect_ratio(rightEye)
            ear = (leftEAR + rightEAR) / 2.0

            leftEyeHull = cv2.convexHull(leftEye)
            rightEyeHull = cv2.convexHull(rightEye)
            cv2.drawContours(frame, [leftEyeHull], -1, (0, 255, 0), 1)
            cv2.drawContours(frame, [rightEyeHull], -1, (0, 255, 0), 1)

            # --- DEBUGGING PRINT ---
            # This line will show you the live values
            cv2.putText(frame, f"EAR: {ear:.2f}", (300, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)


            if ear < thresh:
                flag += 1
                if flag >= frame_check:
                    cv2.putText(frame, "*****ALERT!*****", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    cv2.putText(frame, "*****ALERT!*****", (10, 325),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    display(Audio('alarm.wav', autoplay=True))
            else:
                flag = 0

        cv2_imshow(frame)

        frame = video_frame()
        if frame is None:
            break

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    video_release()
    print("Webcam released.")

--2025-11-26 16:41:59--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2025-11-26 16:41:59--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2.1’

shape_predictor_68_ 100%[===================>]  61.07M  15.2MB/s    in 4.6s    

2025-11-26 16:42:05 (13.2 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2.1’ saved [64040097/64040097]

bzip2: Output file shape_predictor_68_face_landmarks.dat already exists.
--2025-11-26 16:42:05--  https://www.soundjay.com/buttons/sounds/button-3.wav
Resolving www.soundjay.co

<IPython.core.display.Javascript object>

Video stream starting... Please wait for initialization.
An error occurred: Could not start webcam. Please check browser permissions and rerun the cells.
Webcam released.
